# Aggregation method comparison: `mean` vs `simple` vs `evidence`

Two independent lines of evidence toward choosing which behavioral-RDM aggregation method to use (see `analysis/rdms/behavioral.py` for the three methods):

- **Part 1 -- Noise-ceiling reliability**: how much resampled subsets *within* the retained analytical cohort (N=79 pre / 76 post) agree with each other (leave-one-out lower bound, split-half upper bound), computed by `analysis/behavioral/noise_ceiling.py`.
- **Part 2 -- Generalizability**: whether the RDM built from the retained cohort agrees with an independent RDM built from subjects who were never part of it -- pilot-era task versions, screened-out, false-positive -- computed by `analysis/behavioral/generalizability.py`.

A method that's reliable *and* generalizes to independent data is a stronger candidate than one that's merely reliable within its own sample.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]  # analysis/behavioral/ -> repo root
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import wilcoxon

from analysis.behavioral.noise_ceiling import run_comparison, RESULTS_DIR, _DEFAULT_CACHE_DIR
from analysis.utils.stats import wilcoxon_effect_size
from analysis.utils.plotting import role_palette

_TEMPLATE = "plotly_white"  # matches analysis/rdms/imagenet_vs_path.py & hierarchy_comparison.py

## Part 1: Noise-Ceiling Reliability (leave-one-out + split-half)

Compares the three behavioral-RDM aggregation methods via Nili et al. (2014)-style noise-ceiling bounds, computed by `analysis/behavioral/noise_ceiling.py`:

- **Lower bound**: leave-one-participant-out (exact, N=79/76 folds).
- **Upper bound**: split-half (S=100 random draws, same splits reused across all 3 aggregators).
- **Metrics**: Kendall's tau (leading) and Spearman's rho (secondary), computed directly on condensed RDM vectors over jointly-covered pairs. Procrustes/MDS was dropped entirely -- every participant's own trial-derived pair graph is disconnected into ~18 components (confirmed against real data), so a single held-out participant's data can never be MDS-embedded regardless of aggregation method.

This section loads/builds the tidy reliability data, runs the pre-planned pairwise Wilcoxon signed-rank comparisons (evidence-vs-mean primary; evidence-vs-simple, simple-vs-mean secondary), and renders the summary figure.

## Load or build the reliability data

Reuses the cached CSV under `analysis/results/behavioral/` if `noise_ceiling.py`'s CLI has already been run for both cohorts (a full S=100 split-half + N=79/76 LOO sweep over 3 aggregators is nontrivial to re-run -- see the plan's compute-cost notes); otherwise runs it here. Point `cache_dir` at the main checkout's `analysis/results/parsed_data/` if running this notebook from a worktree (that cache isn't copied into worktrees).

In [2]:
cache_dir = _DEFAULT_CACHE_DIR  # override if running from a worktree, e.g. the main checkout's analysis/results/parsed_data
csv_path = RESULTS_DIR / "noise_ceiling.csv"

if csv_path.exists():
    df = pd.read_csv(csv_path)
else:
    df = pd.concat([
        run_comparison("pre_shine", cache_dir=cache_dir, n_splits=100, seed=0),
        run_comparison("post_shine", cache_dir=cache_dir, n_splits=100, seed=0),
    ], ignore_index=True)
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(csv_path, index=False)

df = df[~df["underpowered"]]  # drop any fold that didn't clear min_overlap jointly-covered pairs
df.head()

,cohort,aggregation,bound,fold_id,n_overlap,underpowered,metric,value,ceiling_full
0,pre,mean,lower,6998a40984a21b26ed35074c,2129,False,kendall,0.089705,NaN
1,pre,mean,lower,68598a1d4cebd213b2abb1d9,2240,False,kendall,0.130174,NaN
2,pre,mean,lower,67341b607b7ab78bc222464e,2215,False,kendall,0.144703,NaN
3,pre,mean,lower,69a50d29b53acc0ed832d33f,2205,False,kendall,0.083020,NaN
4,pre,mean,lower,65fbccca15536d60943e0e40,2178,False,kendall,0.190198,NaN


## Summary table: mean ± SEM per (aggregator × metric × cohort × bound)

In [3]:
summary = (
    df.groupby(["cohort", "bound", "metric", "aggregation"])["value"]
    .agg(mean="mean", sem=lambda s: s.std(ddof=1) / np.sqrt(len(s)), n="count")
    .reset_index()
)
summary

,cohort,bound,metric,aggregation,mean,sem,n
0,post,lower,kendall,evidence,0.125402,0.003778,76
1,post,lower,kendall,mean,0.137639,0.004278,76
2,post,lower,kendall,simple,0.138984,0.004299,76
3,post,lower,spearman,evidence,0.187049,0.005577,76
4,post,lower,spearman,mean,0.205116,0.006321,76
5,post,lower,spearman,simple,0.207041,0.006348,76
6,post,upper,kendall,evidence,0.119321,0.000234,100
7,post,upper,kendall,mean,0.138662,0.000254,100
8,post,upper,kendall,simple,0.140392,0.000252,100
9,post,upper,spearman,evidence,0.178258,0.000339,100


## Spearman-Brown-projected split-half ceiling (`ceiling_full`)

The raw split-half Spearman (`bound='upper'`) is also projected to full-sample size via the Spearman-Brown correction `2r/(1+r)` (mirroring `SpAM_Simulations/empirical/gt_diagnostics.py`'s `raw_noise_ceiling()`), for direct comparability with that module's own prior real-data noise-ceiling numbers (`SpAM_Simulations/sim_results/v5/prod/prod_noise_ceiling.csv`: pre `ceiling_half`=0.191, `ceiling_full`=0.321; post `ceiling_half`=0.171, `ceiling_full`=0.292 -- computed with `aggregation='mean'`-equivalent pooling, so the `mean` row below is the most directly comparable). Not computed for Kendall (the correction isn't well-justified for a rank-concordance statistic) or for the lower bound (LOO already uses N-1 participants, not a half, so there's no half-to-full projection to make).

In [4]:
ceiling = (
    df[(df["bound"] == "upper") & (df["metric"] == "spearman")]
    .groupby(["cohort", "aggregation"])
    .agg(raw_ceiling_half=("value", "mean"), ceiling_full=("ceiling_full", "mean"))
    .reset_index()
)
ceiling

,cohort,aggregation,raw_ceiling_half,ceiling_full
0,post,evidence,0.178258,0.302564
1,post,mean,0.206779,0.342680
2,post,simple,0.209241,0.346056
3,pre,evidence,0.164327,0.282251
4,pre,mean,0.192630,0.323018
5,pre,simple,0.196730,0.328765


## Pairwise Wilcoxon signed-rank comparisons

Pre-planned pairwise comparisons per (cohort, bound, metric) group, on matched fold/split indices (paired by `fold_id`, which corresponds to the same held-out participant for the lower bound, or the same shared split for the upper bound -- see `noise_ceiling.py`'s `make_splits()`): **evidence-vs-mean (primary)**, evidence-vs-simple, simple-vs-mean (secondary -- decompose whether rescaling alone (`simple`) or rescaling + magnitude-weighting (`evidence`) drives any difference). Effect size is the matched-pairs rank-biserial correlation. Kendall's tau is the leading metric for the "is `evidence` best" verdict; Spearman is reported alongside as a robustness check.

In [5]:
_COMPARISONS = [("evidence", "mean"), ("evidence", "simple"), ("simple", "mean")]

rows = []
for (cohort, bound, metric), group in df.groupby(["cohort", "bound", "metric"]):
    wide = group.pivot(index="fold_id", columns="aggregation", values="value").dropna()
    for a, b in _COMPARISONS:
        if a not in wide.columns or b not in wide.columns:
            continue
        result = wilcoxon_effect_size(wide[a].to_numpy(), wide[b].to_numpy())
        rows.append({"cohort": cohort, "bound": bound, "metric": metric, "comparison": f"{a}_vs_{b}", **result})

wilcoxon_results = pd.DataFrame(rows)
wilcoxon_results

,cohort,bound,metric,comparison,n,statistic,p_value,effect_size
0,post,lower,kendall,evidence_vs_mean,76,79.0,7.752802e-13,-0.946001
1,post,lower,kendall,evidence_vs_simple,76,60.0,3.762927e-13,-0.958988
2,post,lower,kendall,simple_vs_mean,76,938.0,6.565598e-03,0.358852
3,post,lower,spearman,evidence_vs_mean,76,68.0,5.107536e-13,-0.953520
4,post,lower,spearman,evidence_vs_simple,76,59.0,3.621501e-13,-0.959672
5,post,lower,spearman,simple_vs_mean,76,933.0,6.069611e-03,0.362269
6,post,upper,kendall,evidence_vs_mean,100,0.0,3.896560e-18,-1.000000
7,post,upper,kendall,evidence_vs_simple,100,0.0,3.896560e-18,-1.000000
8,post,upper,kendall,simple_vs_mean,100,114.0,1.134379e-16,0.954851
9,post,upper,spearman,evidence_vs_mean,100,0.0,3.896560e-18,-1.000000


In [6]:
# Primary claim: is 'evidence' best, per the leading metric (Kendall), at both bounds?
wilcoxon_results.query("metric == 'kendall' and comparison == 'evidence_vs_mean'")

,cohort,bound,metric,comparison,n,statistic,p_value,effect_size
0,post,lower,kendall,evidence_vs_mean,76,79.0,7.752802e-13,-0.946001
6,post,upper,kendall,evidence_vs_mean,100,0.0,3.896560e-18,-1.000000
12,pre,lower,kendall,evidence_vs_mean,79,32.0,3.873170e-14,-0.979747
18,pre,upper,kendall,evidence_vs_mean,100,0.0,3.896560e-18,-1.000000


## Figure: reliability by aggregator, bound, metric, and cohort

2×2 grid (rows = metric, cols = cohort), each subplot a grouped bar chart (x = bound, grouped/colored by aggregator), bar height = mean, error bars = SEM.

In [7]:
_METRICS = ["kendall", "spearman"]
_COHORTS = ["pre", "post"]
_AGGREGATIONS = ["mean", "simple", "evidence"]
_BOUNDS = ["lower", "upper"]
_COLORS = {k: v["main"] for k, v in role_palette(_AGGREGATIONS).items()}

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f"{metric.title()} — {cohort}-SHINE" for metric in _METRICS for cohort in _COHORTS],
    shared_yaxes=True,
)

for row, metric in enumerate(_METRICS, start=1):
    for col, cohort in enumerate(_COHORTS, start=1):
        cell = summary[(summary["metric"] == metric) & (summary["cohort"] == cohort)]
        for aggregation in _AGGREGATIONS:
            sub = cell[cell["aggregation"] == aggregation].set_index("bound").reindex(_BOUNDS)
            fig.add_trace(
                go.Bar(
                    x=_BOUNDS, y=sub["mean"], error_y=dict(type="data", array=sub["sem"]),
                    name=aggregation, marker_color=_COLORS[aggregation],
                    legendgroup=aggregation, showlegend=(row == 1 and col == 1),
                ),
                row=row, col=col,
            )

fig.update_layout(
    title="Noise-ceiling reliability: mean vs simple vs evidence"
          "<br><sub>lower bound = leave-one-out; upper bound = split-half (S=100)</sub>",
    barmode="group", template=_TEMPLATE, height=760, width=980,
    legend=dict(orientation="h", y=-0.08, x=0.5, xanchor="center"),
)
fig.show()

## Part 2: Generalizability (out-of-sample validation)

Does the analytical RDM (retained cohort) agree with an RDM built from subjects who were **never** part of it? The out-of-sample (OOS) pool is: NOT in the retained cohort (`status=='full data'` & `task_version>=4.0`) AND `status` not in `{'revoked consent', 'missing data'}` (hard exclusions regardless of purpose -- revoked consent is an ethical/legal boundary, not a quality one). This pulls in pilot-era subjects (`task_version` 1.0/2.0/3.0/3.06, `status=='full data'`) plus v4+ subjects with `status` in `{'screened out', 'false_positive'}`.

Real OOS pool sizes (verified against the cached data):

| cohort | OOS N | non-catch trials | pair coverage | connected? |
|---|---|---|---|---|
| pre-SHINE | 63 (42 pilot + 11 screened-out + 10 false-positive) | 888 | 43.22% | yes, 1 component |
| post-SHINE | 18 (6 pilot + 7 screened-out + 5 false-positive) | 286 | 15.65% | yes, 1 component |

Post-SHINE's much sparser coverage (well below the ~39-40% floor already validated elsewhere in this project) means its results below -- especially the MDS leg -- should be read as **lower-confidence**, not suppressed.

**Structural difference from Part 1**: there is exactly **one** analytical RDM and **one** OOS RDM per (cohort, aggregation) -- no resampling, so this is a single point-estimate comparison per method, not a distribution of paired fold values. No Wilcoxon test is meaningful here; "which method generalizes best" is decided by directly comparing the point values, the way a single held-out test-set score is compared across models in ML.

In [8]:
from analysis.behavioral.generalizability import (
    run_generalizability_comparison, run_generalizability_mds_comparison,
    RESULTS_DIR as GEN_RESULTS_DIR,
)

gen_csv_path = GEN_RESULTS_DIR / "generalizability.csv"
if gen_csv_path.exists():
    gen_df = pd.read_csv(gen_csv_path)
else:
    gen_df = run_generalizability_comparison(cache_dir=cache_dir)
    GEN_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    gen_df.to_csv(gen_csv_path, index=False)

gen_df = gen_df[~gen_df["underpowered"]]
gen_df.head()

,cohort,aggregation,n_oos_participants,n_oos_trials,pct_oos_coverage,oos_status_breakdown,n_overlap,underpowered,metric,value
0,pre,mean,63,888,43.224614,"{'full data': 42, 'screened out': 11, 'false_p...",73594,False,kendall,0.117755
1,pre,evidence,63,888,43.224614,"{'full data': 42, 'screened out': 11, 'false_p...",73594,False,kendall,0.101536
2,pre,simple,63,888,43.224614,"{'full data': 42, 'screened out': 11, 'false_p...",73594,False,kendall,0.121774
3,post,mean,18,286,15.645647,"{'screened out': 7, 'full data': 6, 'false_pos...",25976,False,kendall,0.098486
4,post,evidence,18,286,15.645647,"{'screened out': 7, 'full data': 6, 'false_pos...",25976,False,kendall,0.073736


### Raw RDM generalizability (Kendall/Spearman)

One row per (cohort, aggregation, metric) -- no mean/SEM, since there's exactly one comparison per cell (see the structural note above).

In [9]:
raw_gen_table = gen_df.pivot_table(
    index=["cohort", "aggregation"], columns="metric", values="value"
).reset_index()
raw_gen_table

metric,cohort,aggregation,kendall,spearman
0,post,evidence,0.073736,0.110399
1,post,mean,0.098486,0.147083
2,post,simple,0.099791,0.149103
3,pre,evidence,0.101536,0.151437
4,pre,mean,0.117755,0.175498
5,pre,simple,0.121774,0.181397


### MDS-embedding generalizability (Procrustes)

Unlike Part 1 (where a single held-out participant's own trial graph is provably disconnected, so MDS can never run there), both OOS pools here are pooled, multi-subject data and confirmed single connected components -- MDS is feasible, and cheap enough (a handful of one-time R fits, not the S=100-iteration budget Part 1's split-half leg would have needed) to sweep `ndim ∈ {2, 3, 5, 10}`, matching the earlier R-vs-rsatoolbox deep-dive's grid (`analysis/rsatoolbox_eval/`).

This needs a live R/rpy2/`smacof` installation (`R_LIBS_USER` set to the local R library). The next cell probes for it and sets `R_AVAILABLE` so the notebook stays re-runnable on a machine without R configured -- it skips cleanly with a message instead of crashing, rather than requiring R everywhere this notebook is opened.

In [10]:
try:
    from SpAM_Simulations.core.multi_dimensional_scaling import run_mds  # noqa: F401
    R_AVAILABLE = True
    print("R backend available -- MDS generalizability leg will run.")
except Exception as e:
    R_AVAILABLE = False
    print(f"R backend not available ({e!r}) -- skipping the MDS generalizability leg.")

R backend available -- MDS generalizability leg will run.


In [11]:
mds_csv_path = GEN_RESULTS_DIR / "generalizability_mds.csv"
if R_AVAILABLE:
    if mds_csv_path.exists():
        mds_df = pd.read_csv(mds_csv_path)
    else:
        mds_df = run_generalizability_mds_comparison(dims=(2, 3, 5, 10), cache_dir=cache_dir)
        GEN_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        mds_df.to_csv(mds_csv_path, index=False)
else:
    mds_df = pd.DataFrame()
mds_df

,cohort,aggregation,ndim,niter_analytical,niter_oos,stress_analytical,stress_oos,needs_more_iters_analytical,needs_more_iters_oos,disparity,confdist_r,verdict
0,pre,mean,2,193.0,343.0,0.458594,0.488963,False,False,0.778728,0.450609,DIFFERENT
1,pre,mean,3,498.0,500.0,0.411474,0.441120,False,True,0.525328,0.560601,DIFFERENT
2,pre,mean,5,367.0,500.0,0.382475,0.408879,False,True,0.610217,0.568217,DIFFERENT
3,pre,mean,10,419.0,453.0,0.367089,0.391001,False,False,0.686818,0.602033,DIFFERENT
4,pre,evidence,2,309.0,342.0,0.466567,0.510426,False,False,0.779225,0.407880,DIFFERENT
5,pre,evidence,3,428.0,500.0,0.419145,0.466107,False,True,0.545163,0.510140,DIFFERENT
6,pre,evidence,5,414.0,465.0,0.386850,0.437091,False,False,0.650494,0.502977,DIFFERENT
7,pre,evidence,10,479.0,449.0,0.369423,0.420486,False,False,0.700409,0.529736,DIFFERENT
8,pre,simple,2,233.0,318.0,0.452660,0.481450,False,False,0.781682,0.454638,DIFFERENT
9,pre,simple,3,500.0,472.0,0.405007,0.435478,True,False,0.508771,0.573933,DIFFERENT


In [12]:
if not mds_df.empty:
    mds_display = mds_df.copy()
    mds_display["confidence"] = mds_display["cohort"].map({"pre": "normal", "post": "LOW (15.65% OOS coverage)"})
    mds_display[["cohort", "aggregation", "ndim", "disparity", "confdist_r", "verdict", "confidence"]]

### Figure: generalizability by aggregator and cohort

In [13]:
fig2 = make_subplots(
    rows=1, cols=2, subplot_titles=["Raw RDM (Kendall's tau)", "MDS embedding (Procrustes disparity, ndim=10, lower=better)"],
)

for aggregation in _AGGREGATIONS:
    sub = gen_df[(gen_df["metric"] == "kendall") & (gen_df["aggregation"] == aggregation)]
    fig2.add_trace(
        go.Bar(x=sub["cohort"], y=sub["value"], name=aggregation, marker_color=_COLORS[aggregation],
               legendgroup=aggregation, showlegend=True),
        row=1, col=1,
    )
    if not mds_df.empty:
        sub_mds = mds_df[(mds_df["aggregation"] == aggregation) & (mds_df["ndim"] == 10)]
        fig2.add_trace(
            go.Bar(x=sub_mds["cohort"], y=sub_mds["disparity"], name=aggregation, marker_color=_COLORS[aggregation],
                   legendgroup=aggregation, showlegend=False),
            row=1, col=2,
        )

fig2.update_layout(
    title="Generalizability: mean vs simple vs evidence",
    barmode="group", template=_TEMPLATE, height=460, width=980,
    legend=dict(orientation="h", y=-0.15, x=0.5, xanchor="center"),
)
fig2.show()

## Overall verdict

In [14]:
def _rank_aggregations(df, value_col, cohort_col="cohort", higher_is_better=True):
    ranked = df.groupby("aggregation")[value_col].mean().sort_values(ascending=not higher_is_better)
    return list(ranked.index)

part1_kendall = df[df["metric"] == "kendall"]
part1_rank = _rank_aggregations(part1_kendall, "value")
part2_rank = _rank_aggregations(gen_df[gen_df["metric"] == "kendall"], "value")

print(f"Part 1 (noise-ceiling, Kendall, both bounds pooled) ranking, best first: {part1_rank}")
print(f"Part 2 (generalizability, Kendall, raw RDM)         ranking, best first: {part2_rank}")

if not mds_df.empty:
    mds_rank = _rank_aggregations(mds_df[mds_df["ndim"] == 10], "disparity", higher_is_better=False)
    print(f"Part 2 (generalizability, MDS Procrustes disparity, ndim=10) ranking, best first: {mds_rank}")

if part1_rank[0] == part2_rank[0]:
    print(f"\nBoth lines of evidence agree: '{part1_rank[0]}' is the best-supported aggregation method.")
else:
    print(f"\nLines of evidence DISAGREE: Part 1 favors '{part1_rank[0]}', Part 2 (raw RDM) favors '{part2_rank[0]}' -- worth a closer look before deciding.")

Part 1 (noise-ceiling, Kendall, both bounds pooled) ranking, best first: ['simple', 'mean', 'evidence']
Part 2 (generalizability, Kendall, raw RDM)         ranking, best first: ['simple', 'mean', 'evidence']
Part 2 (generalizability, MDS Procrustes disparity, ndim=10) ranking, best first: ['simple', 'mean', 'evidence']

Both lines of evidence agree: 'simple' is the best-supported aggregation method.
